# 📊 全球实时行情 APP - APK 一键编译

**零配置，点击运行即可**

编译完成后，APK 文件会自动下载到你的电脑。

预计时间：首次约15-30分钟（需下载SDK）

In [ ]:
# @title ▶ 点击这里，然后点击左边的 ▶ 运行

import os, sys, subprocess, glob, shutil, urllib.request, zipfile, io
from pathlib import Path
from google.colab import files

print("=" * 50)
print("📊 全球实时行情 APP - APK 编译")
print("=" * 50)

# 步骤1: 获取项目源码
print("\n[1/4] 获取项目源码...")

ZIP_URL = "https://github.com/CharlieYzq/MarketPricesApp/archive/refs/heads/main.zip"

# 尝试直接从GitHub下载
try:
    req = urllib.request.Request(ZIP_URL, headers={"User-Agent": "Colab/1.0"})
    resp = urllib.request.urlopen(req, timeout=30)
    zip_data = resp.read()
    with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
        zf.extractall("/content")
    # 找到解压目录
    extracted = [d for d in os.listdir("/content") if d.startswith("MarketPricesApp")]
    if extracted:
        project_dir = f"/content/{extracted[0]}"
    else:
        project_dir = "/content/MarketPricesApp"
    print(f"   ✅ 从GitHub下载成功: {project_dir}")
except Exception as e:
    print(f"   ⚠ GitHub下载失败: {e}")
    print("   请手动上传ZIP文件...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.makedirs("/content/app", exist_ok=True)
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall("/content/app")
    project_dir = "/content/app"
    print(f"   ✅ 上传解压成功: {project_dir}")

os.chdir(project_dir)
print(f"   当前目录: {os.getcwd()}")
print(f"   文件: {os.listdir('.')[:10]}")

In [ ]:
# @title ▶ 点击运行 - 安装编译环境

print("\n[2/4] 安装编译环境...")

# 安装系统依赖
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run([
    "apt-get", "install", "-y", "-qq",
    "git", "zip", "unzip", "openjdk-17-jdk",
    "autoconf", "libtool", "pkg-config",
    "zlib1g-dev", "libncurses5-dev",
    "cmake", "libffi-dev", "libssl-dev",
    "wget", "curl",
], capture_output=True)

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# 安装 Buildozer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
               "buildozer", "cython"], capture_output=True)

print("   ✅ 编译环境就绪")

In [ ]:
# @title ▶ 点击运行 - 开始编译APK

print("\n[3/4] 正在编译APK...")
print("   首次编译需下载 Android SDK/NDK (约1GB)")
print("   预计等待 15-30 分钟，请耐心等待 ☕")
print()

# 运行 buildozer
result = subprocess.run(
    ["buildozer", "android", "debug"],
    capture_output=True, text=True, timeout=7200
)

# 显示最后30行日志
lines = result.stdout.split('\n')
print('\n'.join(lines[-40:]))
if result.stderr:
    err_lines = result.stderr.split('\n')
    print('\n'.join(err_lines[-20:]))

In [ ]:
# @title ▶ 点击运行 - 下载APK

print("\n[4/4] 查找APK文件...")

apk_files = list(Path(".").glob("bin/*.apk")) + list(Path("/content").glob("**/*.apk"))

if apk_files:
    apk = apk_files[0]
    size = os.path.getsize(apk) / (1024 * 1024)
    print(f"   ✅ 编译成功！")
    print(f"   📦 APK: {apk.name}")
    print(f"   📏 大小: {size:.1f} MB")
    print()
    print("   正在下载到你的电脑...")
    files.download(str(apk))
    print("   ✅ 下载完成！")
    print()
    print("=" * 50)
    print("📱 安装说明")
    print("=" * 50)
    print("1. 把APK传到手机")
    print("2. 在手机上打开文件管理器，点击APK安装")
    print("3. 如果提示'未知来源'，在设置中允许")
    print("4. 打开APP即可查看实时行情")
else:
    print("   ❌ 未找到APK文件，编译可能失败")
    print("   请检查上方日志中的错误信息")